# 03_explore_feature_quality

This notebook **does not modify any datasets**. It audits `combined_dataset_standardised.csv` in order to decide:

- Which columns are usable as metadata features
- How much missingness each column has
- How much `"unknown"` each column has (post-standardisation)
- Whether `"unknown"` is spuriously associated with the label (`is_malignant`)
- Whether columns fingerprint the source dataset (`dataset_id`)

In [1]:
from utils.paths import dataset_dir
import pandas as pd
import numpy as np

DATASET_DIR = dataset_dir()
IN_PATH = DATASET_DIR / "combined_dataset_standardised.csv"

df = pd.read_csv(IN_PATH)
print("Loaded:", IN_PATH)
print("Shape:", df.shape)
df.head(3)


Loaded: /Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/combined_dataset_standardised.csv
Shape: (5659, 41)


,is_malignant,diagnosis,dataset_id,patient_global,img_path,age,sex,fitzpatrick,lesion_size_mm,anatomical_site,...,distance,is_control,melanoma_flag,pathology_report,anatomical_site_clean,cli_impres__group,cli_impres__2_group,cli_impres__3_group,race_group,distance_group
0,0.0,nevus,A,A_PAT_1516,pad_images/PAT_1516_1765_530.png,8,unknown,0,-1.0,upper_extremity,...,-1,unknown,unknown,unknown,arm,other_unclassified,other_unclassified,other_unclassified,unknown,NaN
1,1.0,bcc,A,A_PAT_46,pad_images/PAT_46_881_939.png,55,female,3,6.0,head_neck,...,-1,unknown,unknown,unknown,neck,other_unclassified,other_unclassified,other_unclassified,unknown,NaN
2,1.0,ak,A,A_PAT_1545,pad_images/PAT_1545_1867_547.png,77,unknown,0,-1.0,head_neck,...,-1,unknown,unknown,unknown,face,other_unclassified,other_unclassified,other_unclassified,unknown,NaN


## 1) Basic column overview

- `pct_null`: raw missingness
- `n_unique`: cardinality

High missingness and very high cardinality are common reasons to drop a column.


In [2]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(dropna=True),
    "pct_null": (df.isna().mean() * 100).round(2),
}).sort_values(["pct_null", "n_unique"], ascending=[False, False])

summary


,dtype,n_unique,pct_null
distance_group,float64,3,45.01
img_path,object,5659,0.00
patient_global,object,2099,0.00
pathology_report,object,518,0.00
anatomical_site_clean,object,430,0.00
age,int64,87,0.00
lesion_size_mm,float64,78,0.00
diameter_2,float64,39,0.00
clinical_impression,object,15,0.00
clinical_impression_2,object,15,0.00


## 2) How much `"unknown"` exists per column

This audits your specific concern: after standardisation, many NaNs may become `"unknown"`.

**Important:** `"unknown"` is not inherently bad, but it becomes a problem if:
- it appears mostly in one dataset (dataset fingerprinting), or
- it correlates strongly with the label.


In [3]:
def pct_unknown(series: pd.Series) -> float:
    # Treat any case variant of 'unknown' as unknown
    s = series.astype(str).str.strip().str.lower()
    return float((s == "unknown").mean() * 100)

unknown_summary = pd.DataFrame({
    "pct_unknown": df.apply(pct_unknown)
}).sort_values("pct_unknown", ascending=False)

unknown_summary


,pct_unknown
background_mother,73.917653
background_father,73.846969
smoking,73.599576
pesticide,73.599576
has_sewage_system,73.599576
has_piped_water,73.599576
skin_cancer_history,73.599576
cancer_history,73.599576
alcohol_consumption,73.599576
clinical_impression_3,62.731931


## 3) Does `"unknown"` correlate with the label?

For each **categorical** column, we test whether being `unknown` is associated with `is_malignant`.

- Uses a Chi-square test on a 2×2 contingency table (unknown vs not-unknown) × (benign vs malignant)
- Small p-values suggest **spurious correlation** via missingness patterns

Interpretation:
- If `pct_unknown` is high **and** p-value is very small → consider dropping the column or using a missingness-flag strategy.


In [4]:
from scipy.stats import chi2_contingency

LABEL = "is_malignant"
if LABEL not in df.columns:
    raise ValueError(f"Expected label column '{LABEL}' to exist.")

def unknown_label_pvalue(col: str):
    s = df[col]
    # Only meaningful for categoricals / object columns
    if s.dtype != object:
        return np.nan
    mask_unknown = s.astype(str).str.strip().str.lower().eq("unknown")
    if mask_unknown.sum() == 0 or mask_unknown.sum() == len(mask_unknown):
        return np.nan
    ct = pd.crosstab(mask_unknown, df[LABEL])
    # Ensure 2x2
    if ct.shape != (2, 2):
        return np.nan
    chi2, p, _, _ = chi2_contingency(ct)
    return float(p)

pvals = {}
for col in df.columns:
    pvals[col] = unknown_label_pvalue(col)

unknown_assoc = pd.DataFrame({
    "pct_unknown": unknown_summary["pct_unknown"],
    "p_value_unknown_vs_label": pd.Series(pvals),
}).sort_values(["p_value_unknown_vs_label", "pct_unknown"], ascending=[True, False])

unknown_assoc


,pct_unknown,p_value_unknown_vs_label
background_father,73.846969,1.169793e-190
background_mother,73.917653,1.619389e-190
alcohol_consumption,73.599576,7.655455e-187
cancer_history,73.599576,7.655455e-187
has_piped_water,73.599576,7.655455e-187
has_sewage_system,73.599576,7.655455e-187
pesticide,73.599576,7.655455e-187
skin_cancer_history,73.599576,7.655455e-187
smoking,73.599576,7.655455e-187
biopsed,59.392119,6.090391e-129


## 4) Dataset fingerprinting check (optional but recommended)

If a feature’s distribution differs dramatically across `dataset_id`, your Stage2 model may learn dataset identity.

This prints normalized crosstabs for categorical columns.
If `dataset_id` is missing, this section will be skipped.


In [5]:
if "dataset_id" not in df.columns:
    print("No dataset_id column found — skipping fingerprinting checks.")
else:
    CAT_COLS = [c for c in df.columns if df[c].dtype == object and c not in ["img_path", "patient_global"]]
    
    # Only show top categories to avoid huge output
    TOPK = 10
    
    for col in CAT_COLS:
        ct = pd.crosstab(df[col].astype(str), df["dataset_id"], normalize="columns")
        # show only the most frequent values overall
        top_vals = df[col].astype(str).value_counts().head(TOPK).index
        ct = ct.loc[ct.index.intersection(top_vals)]
        if ct.shape[0] == 0:
            continue
        print("\n===", col, "===")
        display(ct)



=== diagnosis ===


dataset_id,A,B
diagnosis,,
ak,0.317668,0.066052
bcc,0.367711,0.184766
benign_other,0.000000,0.209164
melanoma,0.022628,0.106218
nevus,0.106179,0.213032
scc,0.083551,0.113954
seborrheic_keratosis,0.102263,0.106813



=== dataset_id ===


dataset_id,A,B
dataset_id,,
A,1.0,0.0
B,0.0,1.0



=== sex ===


...

dataset_id,A,B
cli_impres__2_group,,
ak,0.0,0.054746
bcc,0.0,0.053258
benign_other,0.0,0.147575
melanoma,0.0,0.136864
nevus,0.0,0.197560
other_malignant,0.0,0.003570
other_unclassified,1.0,0.158286
scc,0.0,0.171378
seborrheic_keratosis,0.0,0.076763



=== cli_impres__3_group ===


dataset_id,A,B
cli_impres__3_group,,
ak,0.0,0.044035
bcc,0.0,0.048795
benign_other,0.0,0.077358
melanoma,0.0,0.156204
nevus,0.0,0.087474
other_malignant,0.0,0.010414
other_unclassified,1.0,0.389170
scc,0.0,0.133889
seborrheic_keratosis,0.0,0.052663



=== race_group ===


dataset_id,A,B
race_group,,
asian,0.0,0.050878
other_minority,0.0,0.058614
unknown,1.0,0.030646
white,0.0,0.859863


## 5) Quick recommendations table (heuristics)

These are **heuristics**, not rules:
- Drop columns with `pct_null > 70%` (very sparse)
- Drop/ablate columns with `pct_unknown > 50%` **and** very small p-value
- Prefer keeping columns where unknown is rare or not label-associated

Use this table to decide your final `STAGE2_FEATURES` list in `04_audit_and_schema.ipynb`.


In [6]:
heur = unknown_assoc.copy()
heur["pct_null"] = summary["pct_null"]
heur["dtype"] = summary["dtype"]
heur["n_unique"] = summary["n_unique"]

# Heuristic flags
heur["flag_sparse_null_gt_70"] = heur["pct_null"] > 70
heur["flag_unknown_gt_50"] = heur["pct_unknown"] > 50
heur["flag_unknown_label_assoc_p_lt_1e-4"] = heur["p_value_unknown_vs_label"] < 1e-4

heur.sort_values(["flag_unknown_label_assoc_p_lt_1e-4", "flag_unknown_gt_50", "flag_sparse_null_gt_70", "pct_unknown"],
                 ascending=[False, False, False, False])


,pct_unknown,p_value_unknown_vs_label,pct_null,dtype,n_unique,flag_sparse_null_gt_70,flag_unknown_gt_50,flag_unknown_label_assoc_p_lt_1e-4
background_mother,73.917653,1.619389e-190,0.00,object,12,False,True,True
background_father,73.846969,1.169793e-190,0.00,object,14,False,True,True
alcohol_consumption,73.599576,7.655455e-187,0.00,object,3,False,True,True
cancer_history,73.599576,7.655455e-187,0.00,object,3,False,True,True
has_piped_water,73.599576,7.655455e-187,0.00,object,3,False,True,True
has_sewage_system,73.599576,7.655455e-187,0.00,object,3,False,True,True
pesticide,73.599576,7.655455e-187,0.00,object,3,False,True,True
skin_cancer_history,73.599576,7.655455e-187,0.00,object,3,False,True,True
smoking,73.599576,7.655455e-187,0.00,object,3,False,True,True
clinical_impression_3,62.731931,1.925305e-21,0.00,object,14,False,True,True


In [10]:
NUMERIC_PLACEHOLDERS = {
    "age": -1,
    "lesion_size_mm": -1,
    "diameter_2": -1,
    "distance": -1,   # distance may be mixed-type; we handle it safely below
}

rows = []
for col, placeholder in NUMERIC_PLACEHOLDERS.items():
    if col not in df.columns:
        rows.append({
            "column": col,
            "placeholder_value": placeholder,
            "present_in_df": False,
            "dtype": None,
            "pct_placeholder": None,
            "min_num": None,
            "max_num": None,
            "n_unique_raw": None,
            "pct_non_numeric": None,
        })
        continue

    s_raw = df[col]
    s_num = pd.to_numeric(s_raw, errors="coerce")  # non-numeric -> NaN

    pct_placeholder = (s_num == placeholder).mean() * 100
    pct_non_numeric = (s_raw.notna() & s_num.isna()).mean() * 100

    rows.append({
        "column": col,
        "placeholder_value": placeholder,
        "present_in_df": True,
        "dtype": str(s_raw.dtype),
        "pct_placeholder": round(pct_placeholder, 2),
        "min_num": None if s_num.dropna().empty else float(s_num.min()),
        "max_num": None if s_num.dropna().empty else float(s_num.max()),
        "n_unique_raw": int(s_raw.nunique(dropna=True)),
        "pct_non_numeric": round(pct_non_numeric, 2),
    })

placeholder_summary = (
    pd.DataFrame(rows)
    .sort_values(["present_in_df", "pct_placeholder"], ascending=[False, False])
    .reset_index(drop=True)
)

placeholder_summary

,column,placeholder_value,present_in_df,dtype,pct_placeholder,min_num,max_num,n_unique_raw,pct_non_numeric
0,diameter_2,-1,True,float64,73.60,-1.0,70.0,39,0.00
1,distance,-1,True,object,40.61,-1.0,-1.0,5,59.39
2,lesion_size_mm,-1,True,float64,14.45,-1.0,100.0,78,0.00
3,age,-1,True,int64,0.00,6.0,99.0,87,0.00


In [11]:
# To also check the standardised distance group feature
df["distance_group"].isna().mean() * 100
# We conclude that around 45% is missing

np.float64(45.00795193497084)

In [14]:
NUMERIC_PLACEHOLDERS = {
    "age": [-1],
    "lesion_size_mm": [-1],
    "diameter_2": [-1],
    "distance": [-1],
}

# For categorical columns
CATEGORICAL_PLACEHOLDERS_DEFAULT = {"unknown"}

# ---- Compute table ----
rows = []
for col in df.columns:
    s = df[col]
    dtype = str(s.dtype)
    n_unique = int(s.nunique(dropna=True))
    pct_missing = float(s.isna().mean() * 100)

    # placeholder mask
    placeholder_mask = pd.Series(False, index=df.index)

    if pd.api.types.is_numeric_dtype(s):
        sentinels = NUMERIC_PLACEHOLDERS.get(col, [])
        if sentinels:
            placeholder_mask = s.isin(sentinels)
    else:
        # object/categorical columns
        s_norm = s.astype(str).str.strip().str.lower()
        placeholder_mask = s_norm.isin(CATEGORICAL_PLACEHOLDERS_DEFAULT)

    pct_placeholder = float(placeholder_mask.mean() * 100)
    pct_missing_or_placeholder = float((s.isna() | placeholder_mask).mean() * 100)

    rows.append({
        "column": col,
        "dtype": dtype,
        "n_unique": n_unique,
        "pct_missing": round(pct_missing, 2),
        "pct_placeholder": round(pct_placeholder, 2),
        "pct_missing_or_placeholder": round(pct_missing_or_placeholder, 2),
    })

missing_placeholder_table = (
    pd.DataFrame(rows)
    .sort_values("pct_missing_or_placeholder", ascending=False)
    .reset_index(drop=True)
)

missing_placeholder_table

,column,dtype,n_unique,pct_missing,pct_placeholder,pct_missing_or_placeholder
0,background_mother,object,12,0.00,73.92,73.92
1,background_father,object,14,0.00,73.85,73.85
2,skin_cancer_history,object,3,0.00,73.60,73.60
3,pesticide,object,3,0.00,73.60,73.60
4,has_sewage_system,object,3,0.00,73.60,73.60
5,has_piped_water,object,3,0.00,73.60,73.60
6,diameter_2,float64,39,0.00,73.60,73.60
7,smoking,object,3,0.00,73.60,73.60
8,alcohol_consumption,object,3,0.00,73.60,73.60
9,cancer_history,object,3,0.00,73.60,73.60


## Final Metadata Feature Decisions (Locked)

This section documents the final decisions regarding which metadata features
are used in the main Stage2 model, based on:

- missingness analysis
- placeholder auditing (`"unknown"`, `-1`)
- leakage risk
- clinical interpretability
- deployment realism

All `"unknown"` values were introduced during standardisation and represent
explicit missingness, not semantic categories.

---

### Features that should be kept in Main Stage2 Model

These features are clinically meaningful, non-leaky, and suitable for
deployment. Placeholder values are treated as missing during preprocessing.

#### Demographics
- `age`
- `sex`
- `fitzpatrick`

#### Lesion Characteristics
- `lesion_size_mm`  
  *Numeric placeholder `-1` → treated as missing with a missingness flag*
- `anatomical_site_clean`

#### Symptoms (Patient-Reported)
The following features may contain `"unknown"` placeholders representing
unrecorded symptoms. These are treated as missing values with explicit
missingness indicators during preprocessing.

- `bleed`
- `hurt`
- `itch`
- `changed`
- `grew`
- `elevation`

#### Medical History
- `cancer_history`
- `skin_cancer_history`

---

### Ablation-Only Features (Not in Main Model)

These features are excluded from the primary model due to high missingness,
contextual bias, or weak clinical justification, but may be evaluated in
ablation studies.

- `distance_group`
- `diameter_2`
- `smoking`
- `alcohol_consumption`
- `pesticide`
- `has_piped_water`
- `has_sewage_system`

---

### Excluded Features (Should not be used for Stage2)

These features are excluded entirely due to leakage, post-diagnostic
information, dataset fingerprinting, or lack of deployment validity.

- `background_father`
- `background_mother`
- `race`
- `race_group`
- `distance`
- `is_control`
- `melanoma_flag`
- `pathology_report`
- `clinical_impression`
- `clinical_impression_2`
- `clinical_impression_3`
- `cli_impres__group`
- `cli_impres__2_group`
- `cli_impres__3_group`
- `biopsed`

---

### Placeholder Handling Policy (Applied in Stage2 Preprocessing)

- Categorical placeholder `"unknown"` → converted to `NaN`
- Numeric placeholder `-1` → converted to `NaN`
- A binary `<feature>_missing` indicator is added for all such features
- Placeholders are never treated as valid categories or numeric values

This ensures that missingness is handled explicitly and prevents artefactual
correlations during model training.